# Notebook 04 — RAG answer-quality evaluation
**Goal:** Measure whether the RAG chain's answers are actually grounded in retrieved content (faithfulness) and actually address the question asked (correctness) — with real numbers, not a vibe check.

By the end of this notebook you will have:
- Run 10 questions grounded in the actual transcripts of the videos currently ingested
- For each: captured the exact retrieved context alongside the answer
- Scored **faithfulness** by showing a judge the real retrieved context and asking it to verify every claim — not just judging plausibility
- Scored **correctness** separately — does the answer address what was asked
- Logged everything to LangSmith and saved results to `data/eval/` with the same archive + summary pattern as the other evals

**What changed from the original version of this notebook:** the previous faithfulness judge only saw the question and the final answer — it was really judging *plausibility* ("does this sound grounded"), not verifiable groundedness. This version shows the judge the actual retrieved text, so it can catch a fluent-sounding answer that quietly drifted from what was actually retrieved.

**Corpus-size caveat — read this before trusting the numbers:** at time of writing there are 5 ingested videos. This is enough to test whether *generation* is honest given whatever gets retrieved, but not enough to prove *retrieval* itself is reliable — with so little content, a retrieval mistake (pulling the wrong video for a question) is entirely possible and won't necessarily show up as a low score if faithfulness to the (wrong) retrieved text is still high. Treat this as a generation-quality baseline, not a retrieval benchmark. Revisit once the corpus grows.

**Prerequisites:** Run notebook 01 first so there's real content to query against.

## Step 1 — Environment check

In [ ]:
import sys, os
sys.path.append('..')

from src.utils.config import OPENAI_API_KEY, LANGCHAIN_API_KEY, LANGCHAIN_PROJECT

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = LANGCHAIN_API_KEY
os.environ['LANGCHAIN_PROJECT'] = LANGCHAIN_PROJECT

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print(f'✅ LangSmith project: {LANGCHAIN_PROJECT}')

## Step 2 — Load the Q&A test set
Each question is grounded in the actual transcript content of the videos ingested
at time of writing — not generic filler questions. See `data/eval/rag_qa_set.json`
for the full list including which video(s) each answer should draw from.

In [ ]:
import json
from pathlib import Path

qa_path = Path('../data/eval/rag_qa_set.json')
qa_data = json.loads(qa_path.read_text(encoding='utf-8'))
eval_examples = qa_data['examples']

print(f'Loaded {len(eval_examples)} questions')
print(f'Ingested videos at time of writing: {len(qa_data["ingested_videos_at_time_of_writing"])}')
for vid, desc in qa_data['ingested_videos_at_time_of_writing'].items():
    print(f'  {vid}: {desc}')

## Step 3 — Run retrieval + generation, capturing the actual context used

In [ ]:
from src.utils.rag_eval import retrieve_and_answer

run_results = []

for i, ex in enumerate(eval_examples, 1):
    print(f'[{i}/{len(eval_examples)}] {ex["question"]}')
    result = retrieve_and_answer(ex['question'])

    retrieval_hit = bool(set(result['retrieved_video_ids']) & set(ex['expected_video_ids']))

    run_results.append({
        'id': ex['id'],
        'question': ex['question'],
        'expected_answer_criteria': ex['expected_answer_criteria'],
        'expected_video_ids': ex['expected_video_ids'],
        'retrieved_video_ids': result['retrieved_video_ids'],
        'retrieval_hit': retrieval_hit,
        'context': result['context'],
        'answer': result['answer'],
    })

    hit_marker = '✓' if retrieval_hit else '✗ (retrieved wrong video)'
    print(f'  Retrieved: {result["retrieved_video_ids"]} {hit_marker}')
    print(f'  Answer: {result["answer"][:150]}...' if len(result['answer']) > 150 else f'  Answer: {result["answer"]}')
    print()

n_retrieval_hits = sum(1 for r in run_results if r['retrieval_hit'])
print(f'✅ Retrieval sanity check: {n_retrieval_hits}/{len(run_results)} questions retrieved from an expected video.')
print('(Directional signal only — see corpus-size caveat above.)')

## Step 4 — Judge faithfulness (grounded in actual retrieved context)
The judge sees the real retrieved text, not just the question — it verifies each
claim in the answer against what was actually retrieved.

In [ ]:
from src.utils.rag_eval import judge_faithfulness

for i, r in enumerate(run_results, 1):
    verdict = judge_faithfulness(r['answer'], r['context'])
    r['faithfulness_score'] = verdict['faithfulness_score']
    r['unsupported_claims'] = verdict.get('unsupported_claims', [])
    r['faithfulness_reasoning'] = verdict.get('reasoning', '')
    print(f'[{i}/{len(run_results)}] {r["id"]} — faithfulness {r["faithfulness_score"]}/5')
    if r['unsupported_claims']:
        print(f'    Unsupported: {r["unsupported_claims"]}')

print('\n✅ Faithfulness scoring complete.')

## Step 5 — Judge correctness (does the answer address the question)

In [ ]:
from src.utils.rag_eval import judge_correctness

for i, r in enumerate(run_results, 1):
    verdict = judge_correctness(r['question'], r['expected_answer_criteria'], r['answer'])
    r['correctness_score'] = verdict['correctness_score']
    r['correctness_reasoning'] = verdict.get('reasoning', '')
    print(f'[{i}/{len(run_results)}] {r["id"]} — correctness {r["correctness_score"]}/5')

print('\n✅ Correctness scoring complete.')

## Step 6 — Results summary

In [ ]:
print('='*80)
print('RAG ANSWER-QUALITY EVALUATION RESULTS')
print('='*80)

for i, r in enumerate(run_results, 1):
    print(f'\n--- {r["id"]}: {r["question"]} ---')
    print(f'  Faithfulness: {r["faithfulness_score"]}/5 — {r["faithfulness_reasoning"]}')
    print(f'  Correctness:  {r["correctness_score"]}/5 — {r["correctness_reasoning"]}')
    print(f'  Retrieval hit: {r["retrieval_hit"]}')

n = len(run_results)
avg_faithfulness = sum(r['faithfulness_score'] for r in run_results) / n
avg_correctness = sum(r['correctness_score'] for r in run_results) / n
retrieval_hit_rate = sum(1 for r in run_results if r['retrieval_hit']) / n

print(f'\n{"="*80}')
print(f'AVERAGES ({n} questions):')
print(f'  Faithfulness:      {avg_faithfulness:.2f}/5')
print(f'  Correctness:       {avg_correctness:.2f}/5')
print(f'  Retrieval hit rate: {retrieval_hit_rate:.1%} (directional only — see corpus-size caveat)')
print('='*80)

## Step 7 — Save results (archive + latest, matching the other evals)

In [ ]:
from datetime import datetime, timezone

output = {
    'run_at': datetime.now(timezone.utc).isoformat(),
    'n': n,
    'avg_faithfulness': round(avg_faithfulness, 3),
    'avg_correctness': round(avg_correctness, 3),
    'retrieval_hit_rate': round(retrieval_hit_rate, 3),
    'note': 'Baseline run — first proper faithfulness+correctness split, grounded in real retrieved context.',
    'predictions': run_results,
}

results_path = Path('../data/eval/rag_qa_results.json')
results_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Saved latest results to {results_path}')

runs_dir = Path('../data/eval/runs')
runs_dir.mkdir(parents=True, exist_ok=True)
run_label = output['run_at'].replace(':', '').replace('.', '')[:15]
archive_path = runs_dir / f'{run_label}_rag.json'
archive_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Archived this run to {archive_path}')

## Step 8 — Regenerate the one-place summary

In [ ]:
from src.utils.rag_eval import generate_rag_summary_md

summary_path = generate_rag_summary_md()
print(f'✅ Regenerated {summary_path}')
print(f'\n{summary_path.read_text(encoding="utf-8")}')

## Step 9 — Log to LangSmith
Creates/updates a dated dataset with the full question, context, answer, and both
scores per example — browsable at [smith.langchain.com](https://smith.langchain.com).

In [ ]:
from langsmith import Client

ls_client = Client(api_key=LANGCHAIN_API_KEY)
dataset_name = f'copilot-rag-eval-{datetime.now().strftime("%Y%m%d")}'

try:
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description='RAG answer-quality eval — faithfulness (grounded in retrieved context) + correctness.',
    )
    print(f'✅ Created LangSmith dataset: {dataset_name}')
except Exception:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f'✅ Using existing LangSmith dataset: {dataset_name}')

for r in run_results:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={'question': r['question']},
        outputs={
            'answer': r['answer'],
            'faithfulness_score': r['faithfulness_score'],
            'correctness_score': r['correctness_score'],
            'retrieval_hit': r['retrieval_hit'],
        },
    )

print(f'\n✅ Logged {len(run_results)} examples to LangSmith.')

## Notes

**Interpreting a low correctness + high faithfulness combination:** this means the
answer was honest about (or consistent with) what was retrieved, but retrieval itself
grabbed the wrong content — a retrieval problem, not a hallucination problem. Check
`retrieved_video_ids` vs `expected_video_ids` per example to confirm.

**Revisit once the corpus grows:** with 20+ videos, re-run this notebook and expect
`retrieval_hit_rate` to become the more informative number — right now, with only 5
videos, most questions have limited competing content to be retrieved instead.

**Re-running:** Safe any time. `rag_qa_results.json` reflects the latest run; every
run is archived under `data/eval/runs/<timestamp>_rag.json` and indexed in
`data/eval/RAG_SUMMARY.md`.

**Extending the test set:** add more questions to `data/eval/rag_qa_set.json` as more
videos get ingested — keep every question grounded in real, checkable transcript content.